# Exercise 4 — Struct-Based Dispatch Model


## Task

Extend the dispatch model so that firms may submit multiple bids. Each bid is represented by

```julia
mutable struct Bid
    ID::String
    cap::Vector{Float64}
    cost::Vector{Float64}
    renewable::Bool
    firm::String
end
```

For each time step, find the cost-minimizing dispatch subject to:

- demand being met **exactly**;
- each bid staying within its offered capacity;
- no firm supplying more than **50% of demand**; and
- renewable generation supplying no more than **90% of demand**.

First formulate the optimization problem algebraically.

Then implement:

1. `get_firms(bids)`, returning the unique firm names in a `Vector`;
2. `gen_dis(bids, demand, optimizer)`, returning the **unsolved** JuMP model;
3. `modify_model(model, timelimit)`, which silences solver output and sets a solution time limit; and
4. `check_model(model)`, which checks whether the model was solved to optimality. If the model is not optimal:
   - throw `error("Model not solved")` when there is no primal solution;
   - issue `@warn "Model not optimal"` when a primal solution is available.

Use the supplied tests to verify the implementation.


## Supplied data and `Bid` definition


In [1]:
using JuMP, HiGHS, Test, MathOptInterface

mutable struct Bid
    ID::String
    cap::Vector{Float64}
    cost::Vector{Float64}
    renewable::Bool
    firm::String
end

bids = [
    Bid(
        "A_1",
        [9.0, 2.0, 1.0, 0.0, 0.0, 0.0],
        1*[0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        false,
        "A",
    ),
    Bid(
        "B_1",
        [2.0, 1.0, 2.0, 0.0, 0.0, 0.0],
        2*[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        false,
        "B",
    ),
    Bid(
        "C_1",
        [2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
        3*[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        false,
        "C",
    ),
    Bid(
        "A_2",
        [0.0, 0.0, 0.0, 1.0, 2.0, 1.0],
        1*[2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
        false,
        "A",
    ),
    Bid(
        "B_2",
        [0.0, 0.0, 0.0, 2.0, 1.0, 2.0],
        2*[2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
        false,
        "B",
    ),
    Bid(
        "C_2",
        [2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
        3*[2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
        false,
        "C",
    ),
    Bid("A_3", [1.0, 1.0, 0.0, 0.0, 0.0, 9.0], 1*[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], true, "A"),
    Bid("B_3", [0.0, 0.0, 1.0, 1.0, 0.0, 9.0], 2*[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], true, "B"),
    Bid("C_3", [0.0, 0.0, 0.0, 0.0, 1.0, 9.0], 3*[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], true, "C"),
]

demand = [1.0, 2.0, 3.0, 1.0, 2.0, 3.0]


6-element Vector{Float64}:
 1.0
 2.0
 3.0
 1.0
 2.0
 3.0

## Reference implementation


In [2]:
function get_firms(bids)
    firms = String[]
    for b in bids
        push!(firms, b.firm)
    end
    return unique(firms)
end

function gen_dis(bids, demand, optimizer)

    firms = get_firms(bids)
    hours = 1:length(demand)

    m = Model(optimizer)
    @variable(m, x[bids, hours] >= 0)

    @objective(m, Min, sum(b.cost[h] * x[b, h] for b in bids, h in hours))

    @expression(m, supply[h in hours], sum(x[b, h] for b in bids))
    @constraint(m, demand_match[h in hours], supply[h] == demand[h])
    @constraint(m, cap_lim[b in bids, h in hours], x[b, h] <= b.cap[h])
    @constraint(
        m,
        max_renewable[h in hours],
        sum(x[b, h] for b in bids if b.renewable) <= 0.9*demand[h]
    )
    @constraint(
        m,
        max_firmshare[f in firms, h in hours],
        sum(x[b, h] for b in bids if b.firm == f) <= 0.5*demand[h]
    )

    return m
end

function modify_model(model, timelimit)
    set_silent(model)
    set_time_limit_sec(model, timelimit)
end

function check_model(model)
    if termination_status(model) != OPTIMAL
        if has_values(model)
            @warn "Model not optimal"
        else
            error("Model not solved")
        end
    end
    return nothing
end


check_model (generic function with 1 method)

## Supplied tests


In [3]:
@assert length(get_firms(bids)) == 3
@assert "A" ∈ get_firms(bids)
@assert "B" ∈ get_firms(bids)
@assert "C" ∈ get_firms(bids)

m = gen_dis(bids, demand, HiGHS.Optimizer)
optimize!(m)
@assert isapprox(objective_value(m), 10.1, atol = 1e-4)

model_2 = Model(HiGHS.Optimizer)
modify_model(model_2, 10)
@assert get_optimizer_attribute(model_2, MOI.Silent()) == true
@assert isapprox(time_limit_sec(model_2), 10.0; atol = 1e-4)

model_3 = Model(
    () -> MathOptInterface.Utilities.MockOptimizer(
        MathOptInterface.Utilities.Model{Float64}(),
    ),
)
@variable(model_3, x, Bin)
optimize!(model_3)
mock = unsafe_backend(model_3)

@test_throws r"[Mm]odel not solved" check_model(model_3)
MOI.set(mock, MOI.TerminationStatus(), MOI.TIME_LIMIT)
MOI.set(mock, MOI.PrimalStatus(), MOI.FEASIBLE_POINT)
MOI.set(mock, MOI.VariablePrimal(), optimizer_index(x), 1.0)
@test_logs (:warn, r"[Mm]odel not optimal") check_model(model_3)

MOI.set(mock, MOI.TerminationStatus(), MOI.OPTIMAL)
@assert isnothing(check_model(model_3))


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 84 rows; 54 cols; 180 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 6e+00]
  Bound   [0e+00, 0e+00]
  RHS     [5e-01, 9e+00]
Presolving model
19 rows, 32 cols, 61 nonzeros 0s
13 rows, 24 cols, 39 nonzeros 0s
Dependent equations search running on 4 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
11 rows, 18 cols, 33 nonzeros 0s
Presolve reductions: rows 11(-73); columns 18(-36); nonzeros 33(-147) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     2.0000000000e+00 Pr: 4(10) 0.0s
          8     1.0100000000e+01 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve
Using